<a id="vlm-sandbox-models"></a>
# VideoDB Understanding: VLM with Sandbox Models

Run Qwen or Gemma vision-language models on a compatible VideoDB sandbox.
> **Preview status:** Managed Understanding passes end to end. Current Sandbox VLM routing can return per-scene model errors or untimestamped output; this notebook validates both conditions and stops instead of presenting an unusable artifact as success.


<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/vlm/sandbox-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install and connect

Sandbox Compute runs private/open-weight models. Provision a compatible sandbox in the VideoDB Console, set `VIDEODB_SANDBOX_ID`, and stop it in the Console when finished. Sandboxes are billable.

In [ ]:
!pip install -q "videodb>=0.5.0" python-dotenv

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()

print("Connected to VideoDB")
print(f"Collection: {collection.id}")

## 2. Choose a video

In [ ]:
VIDEO_URL = os.getenv(
    "VIDEODB_VIDEO_URL",
    "https://www.youtube.com/watch?v=vVlEVRKv4is",  # Silicon Valley - Gilfoyle is free for hire
)

collection = conn.get_collection()
video = collection.upload(url=VIDEO_URL)

# To reuse a video already uploaded to your account, comment out the upload above
# and use your own VideoDB video ID:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Collection:", collection.id)
print("Video:", video.id)
video.play()


<a id="models"></a>
## 3. Choose a sandbox VLM

| Tier | Example vision models |
|---|---|
| Small | `Qwen/Qwen3.5-9B`, `google/gemma-4-E2B-it` |
| Medium | `Qwen/Qwen3.5-27B`, `google/gemma-4-26B-A4B-it`, `google/gemma-4-31B-it` |

The requested model must be compatible with the sandbox tier.

In [ ]:
VLM_MODEL = "google/gemma-4-E2B-it"  # Fast first choice for a small sandbox
SANDBOX_ID = os.getenv("VIDEODB_SANDBOX_ID")

if not SANDBOX_ID:
    raise ValueError(
        "Set VIDEODB_SANDBOX_ID to an active compatible sandbox from the VideoDB Console."
    )

print("Sandbox configuration:")
print(f"- ID: {SANDBOX_ID}")
print(f"- Model: {VLM_MODEL}")

## 4. Verify the sandbox prerequisite

The published SDK currently supports routing Understanding work to a sandbox but does not expose sandbox lifecycle methods. This guide accepts an active sandbox ID and never provisions billable compute implicitly.

In [ ]:
print("Using active sandbox:", SANDBOX_ID)

## 5. Run the VLM

Supplying both the canonical sandbox model string and `sandbox_id` selects the authorized Open Compute destination. Five 20-second segments with four frames each keep the default validation run bounded.

In [ ]:
understanding = video.understand(
    analyzers=[{
        "type": "vlm",
        "name": "scene",
        "sampling": {"strategy": "uniform", "frame_count": 4},
        "config": {
            "model": VLM_MODEL,
            "sandbox_id": SANDBOX_ID,
            "prompt": "Describe the people, objects, actions, and setting in this scene.",
        },
    }],
    segmentation={"type": "time", "seconds": 20},
)

understanding.wait_until_complete(timeout=3600, poll_interval=15)
print("Understanding complete")
print(f"ID: {understanding.id}")
print(f"Status: {understanding.status}")

## 6. Inspect output

In [ ]:
from pprint import pprint

scene_output = understanding.get_analyzer("scene").get_output()
scenes = scene_output.get("scenes", [])

failed_scenes = [
    scene for scene in scenes
    if (scene.get("data") or {}).get("error")
]
if failed_scenes:
    first_error = (failed_scenes[0].get("data") or {}).get("error")
    raise RuntimeError(f"Sandbox VLM scene processing failed: {first_error}")

invalid_scenes = [
    scene for scene in scenes
    if scene.get("start") is None or scene.get("end") is None
]
if invalid_scenes:
    raise RuntimeError(
        "Sandbox VLM returned scenes without timestamps; the artifact cannot be indexed safely."
    )

print(f"Previewing {min(len(scenes), 5)} of {len(scenes)} scenes")
print("-" * 60)
for scene in scenes[:5]:
    print(f"\n{scene.get('start')}s → {scene.get('end')}s")
    pprint(scene.get("data") or {}, width=100, sort_dicts=False)

## Common failures

- **No active compatible sandbox:** create the correct tier and wait for `active`.
- **Incompatible tier:** select a model listed for the sandbox tier.
- **Unknown model:** use a model exposed by the Sandbox model catalogue.
- **Stopped sandbox:** provision or activate a usable sandbox before submitting.

Understanding never provisions a billable sandbox implicitly.

## Related guides

- [VLM with Managed Models](managed-models.ipynb)
- [Understanding guide map](../README.md)


## 7. Cleanup

Delete the Understanding only when you no longer need its artifact. Stop the billable sandbox separately in the VideoDB Console.

In [ ]:
DELETE_RUN = False

if DELETE_RUN:
    understanding.delete()
    print("Deleted", understanding.id)
else:
    print("Keeping the Understanding artifact. Stop the sandbox in the VideoDB Console.")